# 02 — Model-Independent P(k) MCMC Sampling: DESI 6-Sky Mock (GPU)

Runs full posterior inference on the model-independent parametrisation and compares
four information pathways:

- **(a) P(k)-only MCMC** — sample (ω_cdm, ln10As, h) via Gaussian likelihood built from P(k) chain
- **(b) Growth-only MCMC** — same via growth chain
- **(c) Combined** — joint P(k) + growth Gaussian likelihood
- **(d) Direct** — standard EFT MCMC on (ω_cdm, ln10As, h, EFT)

**Step 1:** run `scripts/generate_fake_data.py`  
**Step 2 (optional):** run `01_fisher_information.ipynb` for comparison Fisher contours  
**Step 3:** run this notebook on a GPU node

GPU tip: Run `sbatch` with `#SBATCH --gpus=1` and set `JAX_PLATFORMS=gpu` below.

In [2]:
import os
# Set GPU mode before importing JAX.
# Switch to 'cpu' for debugging or if no GPU is available.
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["CUDA_DEVICE_ORDER"]         = "PCI_BUS_ID"

import sys
sys.path.insert(0, '../scripts')

import numpy as np
import matplotlib.pyplot as plt
import yaml, emcee
import jax
import jax.numpy as jnp
import jax.scipy as jsp
from scipy.interpolate import interp1d
from getdist import MCSamples, plots

jax.config.update("jax_enable_x64", True)
print(f"JAX devices: {jax.devices()}")

from pybird import config as pybird_config
pybird_config.set_jax_enabled(True)
from pybird.module import *
from pybird.likelihood import Likelihood
from pybird.fake import Fake
from pybird.symbolic import Symbolic
from pybird.symbolic_pofk_linear import plin_emulated

from utils import DESI_Y6, to_Mpc, to_Mpc_jax, to_Mpc_per_h_jax

plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14

PROJECT_ROOT = os.path.abspath('..')
OUTPUT_DIR   = os.path.join(PROJECT_ROOT, 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

JAX devices: [CudaDevice(id=0)]
jax: on


## 1. Setup (Same as Notebook 01)

In [3]:
cosmo_fid = {
    'omega_b':      0.02235,
    'omega_cdm':    0.120,
    'h':            0.675,
    'ln10^{10}A_s': 3.044,
    'n_s':          0.965,
}
h_fid  = cosmo_fid['h']
z_ref  = 3.0

s            = DESI_Y6
num_skies    = s['n_sky']
zeff_list    = s['zeff']
zeff_unique  = sorted(set(zeff_list))
n_z_unique   = len(zeff_unique)
sky_to_z_idx = [zeff_unique.index(z) for z in zeff_list]

M_sym = Symbolic()
M_sym.set(cosmo_fid)
kk_tmp = np.logspace(-4, np.log10(0.7), 200)
M_sym.compute(kk_tmp, 0.5)
Omega0_m = M_sym.c['Omega_m']

print(f"DESI Y6: {num_skies} skies, {n_z_unique} unique z_eff = {zeff_unique}")
print(f"JAX device: {jax.devices()[0]}")

DESI Y6: 7 skies, 6 unique z_eff = [0.295, 0.51, 0.706, 0.93, 1.317, 1.491]
JAX device: cuda:0


In [4]:
# P(k) knots
n_knots = 50
k_min, k_max, k_mid = 0.0005, 0.35, 0.05
k_low  = np.logspace(np.log10(k_min), np.log10(k_mid), n_knots - 20, endpoint=False)
k_high = np.logspace(np.log10(k_mid), np.log10(k_max), 20)
knots_h       = np.concatenate([k_low, k_high])
knots_mpc     = knots_h / h_fid
knots_h_jax   = jnp.array(knots_h)
knots_mpc_jax = jnp.array(knots_mpc)

M_sym.compute(knots_h, z_ref)
pk_at_knots_h   = np.array(M_sym.pk_lin)
pk_at_knots_mpc = to_Mpc(pk_at_knots_h, knots_h, h_fid, knots_mpc)
pk_at_knots_jax = jnp.array(pk_at_knots_mpc)
D_ref           = float(M_sym.D)

# Growth layout
n_growth_per_z  = 3
n_growth_ratios = n_z_unique
n_growth        = n_z_unique * n_growth_per_z + n_growth_ratios + 1

growth_fid_list = []
f_fid, H_fid, DA_fid, D1_fid = {}, {}, {}, {}
for z in zeff_unique:
    M_sym.compute(knots_h, z)
    f_fid[z]  = float(M_sym.f)
    H_fid[z]  = float(M_sym.H)
    DA_fid[z] = float(M_sym.DA)
    D1_fid[z] = float(M_sym.D)
    growth_fid_list.extend([f_fid[z], H_fid[z], DA_fid[z]])
for z in zeff_unique:
    growth_fid_list.append(D1_fid[z] / D_ref)
growth_fid_list.append(h_fid)
growth_fid = np.array(growth_fid_list)

# Likelihood
config_path = os.path.join(OUTPUT_DIR, 'fake_desi_6sky_likelihood_config.yaml')
lkl_config  = yaml.full_load(open(config_path))
lkl_config['drop_logdet'] = True
lkl_config['get_maxlkl']  = True
L_jax = Likelihood(lkl_config)

eft_free_names = ['b1', 'b2', 'b4']
b1_fid = lkl_config['eft_prior']['b1']['mean'][0]
b2_fid = lkl_config['eft_prior']['b2']['mean'][0]
b4_fid = lkl_config['eft_prior']['b4']['mean'][0]
eft_init = np.array([b1_fid, b2_fid, b4_fid] * num_skies)
n_eft    = len(eft_init)

params_fid = np.concatenate([eft_init, np.ones(n_knots), growth_fid])
print(f"params_fid length: {len(params_fid)}")

reading data file: /cluster/work/refregier/alexree/pybird_model_independent/output/fake_desi_6sky.h5
-----------------------
sky: sky_1
output: bPk
multipole: 3
min bound (per multipole): [0.01, 0.01, 0.01]
max bound (per multipole): [0.2, 0.2, 0.2]
coordinate (AP) distortion: on
-----------------------
-----------------------
sky: sky_2
output: bPk
multipole: 3
min bound (per multipole): [0.01, 0.01, 0.01]
max bound (per multipole): [0.2, 0.2, 0.2]
coordinate (AP) distortion: on
-----------------------
-----------------------
sky: sky_3
output: bPk
multipole: 3
min bound (per multipole): [0.01, 0.01, 0.01]
max bound (per multipole): [0.2, 0.2, 0.2]
coordinate (AP) distortion: on
-----------------------
-----------------------
sky: sky_4
output: bPk
multipole: 3
min bound (per multipole): [0.01, 0.01, 0.01]
max bound (per multipole): [0.2, 0.2, 0.2]
coordinate (AP) distortion: on
-----------------------
-----------------------
sky: sky_5
output: bPk
multipole: 3
min bound (per multipol

## 2. Model-Independent Likelihood (GPU-Accelerated)

In [5]:
def model_independent_loglkl(params):
    eft_params    = params[:n_eft]
    pk_amps       = params[n_eft:n_eft + n_knots]
    growth_params = params[n_eft + n_knots:]

    f_z, H_z, DA_z = [], [], []
    for i_z in range(n_z_unique):
        idx = i_z * n_growth_per_z
        f_z.append(growth_params[idx])
        H_z.append(growth_params[idx + 1])
        DA_z.append(growth_params[idx + 2])

    ratio_start = n_z_unique * n_growth_per_z
    D_ratios    = [growth_params[ratio_start + i] for i in range(n_z_unique)]
    h_conv      = growth_params[-1]

    pk_knots_mpc   = pk_amps * pk_at_knots_jax
    pk_knots_h_arr = to_Mpc_per_h_jax(pk_knots_mpc, knots_mpc_jax, h_conv, knots_h_jax)

    cosmo_list = []
    for i_sky in range(num_skies):
        i_z = sky_to_z_idx[i_sky]
        cosmo_list.append({
            'H':      H_z[i_z],
            'DA':     DA_z[i_z],
            'f':      f_z[i_z],
            'pk_lin': pk_knots_h_arr * D_ratios[i_z]**2,
            'kk':     knots_h_jax,
        })

    return L_jax.loglkl(eft_params, eft_free_names * num_skies,
                        need_cosmo_update=True, cosmo_dict=cosmo_list,
                        cosmo_module=None, cosmo_engine=None)


model_independent_loglkl_jit = jax.jit(model_independent_loglkl)

# Sanity check
lkl_fid = float(model_independent_loglkl_jit(jnp.array(params_fid)))
print(f"Log-likelihood at fiducial: {lkl_fid:.6f}  (expect 0.0)")

Log-likelihood at fiducial: -17.346245  (expect 0.0)


## 3. MCMC 1 — Model-Independent Sampling with emcee

96-dim chain: EFT (21) + P(k) amps (50) + growth (25).  
Uses `vectorize=True` with `jax.vmap` so emcee sends a batch of walker positions and
gets log-probs back in one GPU call.

In [6]:
def log_prior(params):
    pk_amps = params[n_eft:n_eft + n_knots]
    growth  = params[n_eft + n_knots:]
    # Positivity: P(k) amps, growth rates, h_conv
    valid = jnp.all(pk_amps > 0) & jnp.all(growth > 0)
    return jnp.where(valid, 0.0, -jnp.inf)


def log_probability(params):
    lp = log_prior(params)
    return jnp.where(jnp.isfinite(lp), lp + model_independent_loglkl_jit(params), -jnp.inf)


# Vectorised over walkers for GPU batching
vec_log_prob = jax.jit(jax.vmap(log_probability))


n_dim_mi    = len(params_fid)      # 96
n_walkers_mi= n_dim_mi * 2 + 2    # 194
n_burn_mi   = 20000
n_steps_mi  = 200000

rng = np.random.default_rng(2026)
init_pos_mi = params_fid + 1e-3 * rng.standard_normal((n_walkers_mi, n_dim_mi))
init_pos_mi[:, n_eft:n_eft + n_knots] = np.abs(init_pos_mi[:, n_eft:n_eft + n_knots])
init_pos_mi[:, n_eft + n_knots:] = np.abs(init_pos_mi[:, n_eft + n_knots:])

print(f"n_dim={n_dim_mi}, n_walkers={n_walkers_mi}")
print(f"burn={n_burn_mi}, steps={n_steps_mi}")
print("GPU note: vmap batches all walkers into a single device call.")

n_dim=96, n_walkers=194
burn=20000, steps=200000
GPU note: vmap batches all walkers into a single device call.


In [7]:
mi_chain_path = os.path.join(OUTPUT_DIR, 'model_independent_chain.npz')

if not os.path.exists(mi_chain_path):
    print("Running model-independent emcee (this will take hours on CPU, ~1h on GPU)...")
    sampler_mi = emcee.EnsembleSampler(
        n_walkers_mi, n_dim_mi, vec_log_prob, vectorize=True
    )
    # Burn-in
    state = sampler_mi.run_mcmc(init_pos_mi, n_burn_mi, progress=True)
    sampler_mi.reset()
    # Production
    sampler_mi.run_mcmc(state, n_steps_mi, progress=True)
    mi_chain_flat = sampler_mi.get_chain(flat=True)
    np.savez(mi_chain_path, chain=mi_chain_flat)
    print(f"Saved {mi_chain_flat.shape[0]} samples to {mi_chain_path}")
else:
    print(f"Loading existing chain from {mi_chain_path}")
    _d = np.load(mi_chain_path)
    mi_chain_flat = _d['chain']
    print(f"Chain shape: {mi_chain_flat.shape}")

Running model-independent emcee (this will take hours on CPU, ~1h on GPU)...


 14%|█▍        | 2751/20000 [06:01<37:44,  7.62it/s] 


ValueError: Probability function returned NaN

## 4. Extract P(k) and Growth Posteriors

In [ ]:
pk_amp_samples  = mi_chain_flat[:, n_eft:n_eft + n_knots]
growth_samples  = mi_chain_flat[:, n_eft + n_knots:]

ratio_start    = n_z_unique * n_growth_per_z
D_ratio_samples = growth_samples[:, ratio_start:ratio_start + n_z_unique]
h_conv_samples  = growth_samples[:, -1]

# P(k) credible envelopes in Mpc^3
pk_recon_mpc = pk_amp_samples * pk_at_knots_mpc[None, :]
pk_med  = np.percentile(pk_recon_mpc, 50,  axis=0)
pk_lo68 = np.percentile(pk_recon_mpc, 16,  axis=0)
pk_hi68 = np.percentile(pk_recon_mpc, 84,  axis=0)
pk_lo95 = np.percentile(pk_recon_mpc, 2.5, axis=0)
pk_hi95 = np.percentile(pk_recon_mpc, 97.5,axis=0)

fig, ax = plt.subplots(figsize=(9, 5))
ax.fill_between(knots_mpc, pk_lo95, pk_hi95, color='C0', alpha=0.20, label='95%')
ax.fill_between(knots_mpc, pk_lo68, pk_hi68, color='C0', alpha=0.40, label='68%')
ax.loglog(knots_mpc, pk_med,       color='C0', lw=2,    label='Median')
ax.loglog(knots_mpc, pk_at_knots_mpc, 'k--', lw=1.5, label='Fiducial')
ax.set_xlabel(r'$k$ [1/Mpc]')
ax.set_ylabel(r'$P_{\rm lin}(k)$ [Mpc$^3$]')
ax.set_title('Reconstructed $P_{\\rm lin}(k)$ from model-independent posterior')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'pk_reconstruction.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"h_conv: {h_conv_samples.mean():.4f} ± {h_conv_samples.std():.4f}  (truth: {h_fid})")

## 5. Build Gaussian Likelihoods from MI Posterior

In [ ]:
# Summary statistics from the MI chain
pk_mean = pk_amp_samples.mean(axis=0)
pk_cov  = np.cov(pk_amp_samples, rowvar=False) + 1e-8 * np.eye(n_knots)

growth_subset    = np.column_stack([D_ratio_samples, h_conv_samples])
growth_mean      = growth_subset.mean(axis=0)
growth_cov       = np.cov(growth_subset, rowvar=False) + 1e-8 * np.eye(growth_subset.shape[1])

combined_obs     = np.column_stack([pk_amp_samples, growth_subset])
comb_mean        = combined_obs.mean(axis=0)
comb_cov         = np.cov(combined_obs, rowvar=False) + 1e-8 * np.eye(combined_obs.shape[1])


def gaussian_loglike_factory(mean_np, cov_np, select_fn):
    """Return a JIT-compiled Gaussian log-likelihood that calls select_fn(theta)."""
    mean_j = jnp.array(mean_np)
    L_chol = jnp.linalg.cholesky(jnp.array(cov_np))

    @jax.jit
    def _loglike(theta):
        obs   = select_fn(theta)
        delta = obs - mean_j
        y     = jsp.linalg.cho_solve((L_chol, True), delta)
        logdet = 2.0 * jnp.sum(jnp.log(jnp.diag(L_chol)))
        return -0.5 * (delta @ y + logdet)

    return _loglike


def cosmo_to_observables(cosmo_params):
    """[omega_cdm, ln10As, h] -> [pk_amps (50), D_ratios (6), h_conv (1)]."""
    omega_cdm, lnAs, h = cosmo_params[0], cosmo_params[1], cosmo_params[2]
    omega_b = cosmo_fid['omega_b']
    n_s     = cosmo_fid['n_s']
    mnu, w0, wa = 0.0, -1.0, 0.0

    A_s     = 1e-10 * jnp.exp(lnAs)
    Omega_m = (omega_cdm + omega_b) / h**2
    Omega_b_h = omega_b / h**2

    pk_h    = 1e9 * plin_emulated(
        knots_h_jax, A_s, Omega_m, Omega_b_h, h, n_s, mnu, w0, wa, a=1.0/(1+z_ref)
    )
    pk_mpc  = to_Mpc_jax(pk_h, knots_h_jax, h, knots_mpc_jax)
    pk_amps = pk_mpc / pk_at_knots_jax

    D_values = []
    for z in zeff_unique:
        D_values.append(D(Omega_m, z, w0, wa))
    D_ref_val = D(Omega_m, z_ref, w0, wa)
    D_ratios  = jnp.array([Dz / D_ref_val for Dz in D_values])

    return jnp.concatenate([pk_amps, D_ratios, jnp.array([h])])


@jax.jit
def select_pk(theta):
    return cosmo_to_observables(theta)[:n_knots]

@jax.jit
def select_growth(theta):
    obs = cosmo_to_observables(theta)
    return obs[n_knots:]   # D_ratios + h_conv

@jax.jit
def select_combined(theta):
    return cosmo_to_observables(theta)   # everything


loglike_pk       = gaussian_loglike_factory(pk_mean,    pk_cov,    select_pk)
loglike_growth   = gaussian_loglike_factory(growth_mean, growth_cov, select_growth)
loglike_combined = gaussian_loglike_factory(comb_mean,  comb_cov,  select_combined)


def make_logprob(loglike_fn):
    @jax.jit
    def _logprob(theta):
        omega_cdm, ln10As, h = theta[0], theta[1], theta[2]
        valid = (omega_cdm > 0.08) & (omega_cdm < 0.14) & (h > 0.4) & (h < 0.8) & (ln10As > 2.8) & (ln10As < 3.5)
        return jnp.where(valid, loglike_fn(theta), -jnp.inf)
    return _logprob

logprob_pk_jax       = make_logprob(loglike_pk)
logprob_growth_jax   = make_logprob(loglike_growth)
logprob_combined_jax = make_logprob(loglike_combined)

print("Gaussian likelihoods built from MI chain.")

## 6. MCMC 2 — Sample Cosmological Parameters from Each Pathway

In [ ]:
cosmo_fid_vec = np.array([cosmo_fid['omega_cdm'], cosmo_fid['ln10^{10}A_s'], cosmo_fid['h']])
cosmo_ndim    = 3
cosmo_nwalkers = 64
cosmo_n_burn  = 5000
cosmo_n_steps = 50000

rng_cosmo = np.random.default_rng(2027)
cosmo_init = cosmo_fid_vec + 1e-3 * rng_cosmo.standard_normal((cosmo_nwalkers, cosmo_ndim))


def run_cosmo_chain(logprob_fn, fname, init=cosmo_init):
    """Run a 3-param cosmological emcee chain."""
    out = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(out):
        print(f"  Loading {fname}")
        return np.load(out)['chain']
    scalar_lp = lambda t: float(logprob_fn(jnp.array(t)))
    s = emcee.EnsembleSampler(cosmo_nwalkers, cosmo_ndim, scalar_lp)
    state = s.run_mcmc(init, cosmo_n_burn, progress=True)
    s.reset()
    s.run_mcmc(state, cosmo_n_steps, progress=True)
    chain = s.get_chain(flat=True)
    np.savez(out, chain=chain)
    print(f"  Saved {fname}: {chain.shape[0]} samples")
    return chain


print("(a) P(k)-only cosmological chain...")
chain_pk = run_cosmo_chain(logprob_pk_jax, 'cosmo_from_pk.npz')

print("(b) Growth-only cosmological chain...")
chain_growth = run_cosmo_chain(logprob_growth_jax, 'cosmo_from_growth.npz')

print("(c) Combined cosmological chain...")
chain_combined = run_cosmo_chain(logprob_combined_jax, 'cosmo_from_combined.npz')

## 7. MCMC 3 — Direct Cosmological Sampling

In [ ]:
def direct_cosmo_loglkl(cosmo_eft_params):
    """Directly evaluates the EFT likelihood for [omega_cdm, ln10As, h, EFT...]."""
    cosmo_p  = cosmo_eft_params[:3]
    eft_p    = cosmo_eft_params[3:]
    obs      = cosmo_to_observables(cosmo_p)
    pk_amps  = obs[:n_knots]
    growth_p = jnp.concatenate([
        jnp.array([f(   (cosmo_p[0]+cosmo_fid['omega_b'])/cosmo_p[2]**2, z, -1., 0.),
                   Hubble((cosmo_p[0]+cosmo_fid['omega_b'])/cosmo_p[2]**2, z, -1., 0.),
                   DA(   (cosmo_p[0]+cosmo_fid['omega_b'])/cosmo_p[2]**2, z, -1., 0.)])
        for z in zeff_unique
    ] + [
        obs[n_knots:]   # D_ratios + h_conv
    ])
    full_p = jnp.concatenate([eft_p, pk_amps, growth_p])
    return model_independent_loglkl_jit(full_p)


direct_loglkl_jit = jax.jit(direct_cosmo_loglkl)

n_dim_direct    = n_eft + cosmo_ndim
n_walkers_direct = 64

rng_d = np.random.default_rng(2028)
init_direct = np.zeros((n_walkers_direct, n_dim_direct))
init_direct[:, :n_eft] = eft_init
init_direct[:, n_eft:] = cosmo_fid_vec
init_direct += 1e-3 * rng_d.standard_normal(init_direct.shape)

direct_path = os.path.join(OUTPUT_DIR, 'cosmo_direct.npz')
if not os.path.exists(direct_path):
    print("(d) Direct cosmological emcee...")
    scalar_direct = lambda t: float(direct_loglkl_jit(jnp.array(t)))
    s_d = emcee.EnsembleSampler(n_walkers_direct, n_dim_direct, scalar_direct)
    state_d = s_d.run_mcmc(init_direct, cosmo_n_burn, progress=True)
    s_d.reset()
    s_d.run_mcmc(state_d, cosmo_n_steps, progress=True)
    chain_direct = s_d.get_chain(flat=True)
    np.savez(direct_path, chain=chain_direct)
    print(f"  Saved cosmo_direct.npz: {chain_direct.shape[0]} samples")
else:
    print(f"Loading {direct_path}")
    chain_direct = np.load(direct_path)['chain']
    print(f"  Chain shape: {chain_direct.shape}")

## 8. Comparison Triangle Plots

In [ ]:
names_3d  = ['omega_cdm', 'ln10As', 'h']
labels_3d = [r'\omega_{\rm cdm}', r'\ln(10^{10}A_s)', r'h']

# Direct chain has n_eft extra columns at the front
chain_direct_cosmo = chain_direct[:, n_eft:]

gd_list = []
for label, chain in [
    ('(d) Direct',       chain_direct_cosmo),
    ('(c) Combined',     chain_combined),
    ('(a) P(k)-only',    chain_pk),
    ('(b) Growth-only',  chain_growth),
]:
    gd_list.append(MCSamples(samples=chain, names=names_3d, labels=labels_3d, label=label))

g = plots.get_subplot_plotter(width_inch=10)
g.triangle_plot(gd_list, params=names_3d,
                filled=[True, False, False, False],
                contour_colors=['#000000', '#43A047', '#E53935', '#1E88E5'],
                legend_labels=[s.label for s in gd_list])
plt.suptitle('Information Decomposition: DESI 6-sky Mock (MCMC)', y=1.01, fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'mcmc_triangle.png'), dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print(f"{'label':<20} {'omega_cdm':>12} {'ln10As':>12} {'h':>12}")
print('-' * 60)
for label, chain in [
    ('(d) Direct',      chain_direct_cosmo),
    ('(c) Combined',    chain_combined),
    ('(a) P(k)-only',   chain_pk),
    ('(b) Growth-only', chain_growth),
]:
    sigmas = chain.std(axis=0)
    print(f"{label:<20} {sigmas[0]:>12.5f} {sigmas[1]:>12.5f} {sigmas[2]:>12.5f}")

## 9. Save All Chains

In [ ]:
np.savez(os.path.join(OUTPUT_DIR, 'cosmo_posteriors.npz'),
         cosmo_from_pk       = chain_pk,
         cosmo_from_growth   = chain_growth,
         cosmo_from_combined = chain_combined,
         cosmo_direct        = chain_direct_cosmo,
         cosmo_fid_vec       = cosmo_fid_vec)
print(f"Saved all chains to {os.path.join(OUTPUT_DIR, 'cosmo_posteriors.npz')}")